# SafeRoute — Démonstration MVP
**Fait par Gillesto**

Ce notebook démontre les 3 itinéraires Pareto-optimaux sur Londres et Le Cap :
- **shortest** : le plus court (distance minimale)
- **safest** : le plus sûr (évite les zones à forte criminalité)
- **balanced** : compromis confort (distance + sécurité + familiarité)

Formule de coût : `C = w1·Distance + w2·Risque - w3·Familiarité`

Algorithme : A*pex biobjectif (Zhang et al., ICAPS 2022)

In [ ]:
import sys
from pathlib import Path

# Ajoute saferoute au path
sys.path.insert(0, str(Path('..') / 'bindings' / 'python'))

import json
import math
import random
import folium
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display

from saferoute.graph_cache import GraphCache
from saferoute.kde_scorer import compute_kde_scores
from saferoute.familiarity import FamiliarityEngine
from saferoute.models import ParetoSet

print('Imports OK')

## 1. Chargement du graphe

Si le cache existe (`data/cache/`), le graphe est chargé en quelques secondes.  
Sinon, lancez d'abord : `python scripts/download_cities.py --offline`

In [ ]:
CITY = 'london'  # 'london' ou 'cape_town'

cache = GraphCache(cache_dir=Path('..') / 'data' / 'cache')

if cache.has_graph(CITY):
    G = cache.load_graph(CITY)
    print(f'Graphe chargé depuis le cache : {G.number_of_nodes()} nœuds, {G.number_of_edges()} arcs')
else:
    print('Cache vide. Génération d\'un graphe synthétique pour la démo...')
    # Graphe synthétique 10×10
    lat0, lon0 = (51.507, -0.127) if CITY == 'london' else (-33.925, 18.424)
    dlat = 0.001
    dlon = 0.001 / math.cos(math.radians(lat0))
    G = nx.MultiDiGraph()
    size = 10
    for i in range(size):
        for j in range(size):
            nid = i * size + j + 1000
            G.add_node(nid, y=lat0 + i * dlat, x=lon0 + j * dlon)
    for i in range(size):
        for j in range(size):
            u = i * size + j + 1000
            if j + 1 < size:
                v = i * size + j + 1 + 1000
                G.add_edge(u, v, key=0, length=111.0)
                G.add_edge(v, u, key=0, length=111.0)
            if i + 1 < size:
                v = (i + 1) * size + j + 1000
                G.add_edge(u, v, key=0, length=111.0)
                G.add_edge(v, u, key=0, length=111.0)
    print(f'Graphe synthétique : {G.number_of_nodes()} nœuds, {G.number_of_edges()} arcs')

## 2. Données de criminalité et scoring KDE

In [ ]:
if cache.has_crimes(CITY):
    crimes = cache.load_crimes(CITY)
    print(f'Crimes chargés depuis le cache : {len(crimes)} points')
else:
    print('Génération de crimes synthétiques...')
    from saferoute.data_loader import fetch_cape_town_crimes
    rng = random.Random(42)
    nodes = list(G.nodes(data=True))
    # Hotspot dans le coin supérieur gauche
    hotspot = nodes[:len(nodes)//4]
    crimes = []
    for _ in range(200):
        _, d = rng.choice(hotspot)
        crimes.append({'lat': rng.gauss(d['y'], 0.0005),
                       'lon': rng.gauss(d['x'], 0.0005),
                       'weight': rng.uniform(1.0, 3.0)})
    print(f'{len(crimes)} crimes synthétiques générés')

# Calcul KDE
kde_result = compute_kde_scores(G, crimes)
print(f'KDE terminé en {kde_result.elapsed_s:.2f}s | bandwidth={kde_result.bandwidth_m:.0f}m')
print(f'Scores : mean={kde_result.stats["mean"]:.3f}, p90={kde_result.stats["p90"]:.3f}, max={kde_result.stats["max"]:.3f}')

## 3. Heatmap de risque

In [ ]:
# Coordonnées des nœuds pour la visualisation
node_pos = {nid: (d['x'], d['y']) for nid, d in G.nodes(data=True)}
lats = [d['y'] for _, d in G.nodes(data=True)]
lons = [d['x'] for _, d in G.nodes(data=True)]
center_lat = (min(lats) + max(lats)) / 2
center_lon = (min(lons) + max(lons)) / 2

# Carte Folium avec heatmap
m = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles='CartoDB positron')

# Arcs colorés par score de risque
for u, v, k, data in G.edges(keys=True, data=True):
    risk = kde_result.risk_map.get((u, v, k), 0.0)
    # Couleur : vert (sûr) → rouge (dangereux)
    r = int(255 * risk)
    g_val = int(255 * (1 - risk))
    color = f'#{r:02x}{g_val:02x}00'
    u_data = G.nodes[u]
    v_data = G.nodes[v]
    folium.PolyLine(
        locations=[[u_data['y'], u_data['x']], [v_data['y'], v_data['x']]],
        color=color, weight=2, opacity=0.6,
        tooltip=f'Risque: {risk:.3f}'
    ).add_to(m)

# Points de crime
for c in crimes[:50]:  # affiche les 50 premiers pour la lisibilité
    folium.CircleMarker(
        location=[c['lat'], c['lon']], radius=3,
        color='darkred', fill=True, fill_opacity=0.5,
        tooltip=f'Crime (poids={c["weight"]:.1f})'
    ).add_to(m)

display(m)

## 4. Simulation de familiarité

In [ ]:
fam_engine = FamiliarityEngine(decay_per_trip=0.02)
fmap = fam_engine.simulate_trajectories(G, n_trips=50, seed=42)
stats = fmap.stats()
print(f'Familiarité simulée : {stats["total_trips"]} trajets')
print(f'Arcs familiers (>0.3) : {stats["familiar_edges"]} / {stats["count"]}')
print(f'Score moyen : {stats["mean"]:.3f} | max : {stats["max"]:.3f}')

## 5. Calcul des 3 itinéraires Pareto-optimaux

> **Note** : cette section nécessite le core Rust compilé (`maturin develop`).  
> Sans Rust, on simule les résultats pour la démonstration visuelle.

In [ ]:
nodes_list = sorted(G.nodes())
SOURCE = nodes_list[0]       # coin supérieur gauche (zone risquée)
TARGET = nodes_list[-1]      # coin inférieur droit (zone sûre)

print(f'Source : {SOURCE} | Target : {TARGET}')

try:
    from saferoute_core import PyGraph, compute_safe_routes

    # Sérialisation du graphe enrichi
    graph_data = {
        'nodes': [{'id': nid, 'lat': d['y'], 'lon': d['x']} for nid, d in G.nodes(data=True)],
        'edges': [{'from': u, 'to': v,
                   'distance_m': float(d.get('length', 100.0)),
                   'risk_score': float(kde_result.risk_map.get((u, v, k), 0.0)),
                   'familiarity': float(fmap.get(u, v, k))}
                  for u, v, k, d in G.edges(keys=True, data=True)]
    }
    graph_json = json.dumps(graph_data)
    py_graph = PyGraph.from_json(graph_json)
    results = compute_safe_routes(py_graph, SOURCE, TARGET, 0.1)
    pareto = ParetoSet.from_results(results)
    print('✅ Core Rust utilisé')

except ImportError:
    print('⚠️  Core Rust non compilé — simulation des résultats pour la démo')
    # Simulation via NetworkX pour la démonstration visuelle
    def _nx_path(src, tgt, weight_fn):
        try:
            return nx.shortest_path(G, src, tgt, weight=weight_fn)
        except nx.NetworkXNoPath:
            return [src, tgt]

    def _path_stats(path):
        dist = sum(G[path[i]][path[i+1]][0].get('length', 100.0) for i in range(len(path)-1))
        risk = sum(kde_result.risk_map.get((path[i], path[i+1], 0), 0.0) for i in range(len(path)-1))
        fam  = sum(fmap.get(path[i], path[i+1], 0) for i in range(len(path)-1))
        return dist, risk, fam

    path_short = _nx_path(SOURCE, TARGET, 'length')
    path_safe  = _nx_path(SOURCE, TARGET,
                          lambda u, v, d: kde_result.risk_map.get((u, v, 0), 0.0) + 0.001)
    path_bal   = _nx_path(SOURCE, TARGET,
                          lambda u, v, d: d.get('length', 100.0) / 1000 +
                                          kde_result.risk_map.get((u, v, 0), 0.0) * 500)

    from saferoute.models import Route
    d1, r1, f1 = _path_stats(path_short)
    d2, r2, f2 = _path_stats(path_safe)
    d3, r3, f3 = _path_stats(path_bal)

    pareto = ParetoSet(
        shortest=Route(path_short, d1, r1, 'shortest', f1, d1/66.67, len(path_short), max(0, 1-r1)),
        safest  =Route(path_safe,  d2, r2, 'safest',   f2, d2/66.67, len(path_safe),  max(0, 1-r2)),
        balanced=Route(path_bal,   d3, r3, 'balanced', f3, d3/66.67, len(path_bal),   max(0, 1-r3)),
    )

# Affichage des résultats
print('\n=== Résultats Pareto ===')
for route in [pareto.shortest, pareto.safest, pareto.balanced]:
    if route:
        print(f'  {route.route_type:10s} | dist={route.total_distance_m:6.0f}m '
              f'| risque={route.total_risk:.3f} '
              f'| temps={route.estimated_time_min:.1f}min '
              f'| confort={route.comfort_score:.2f}')

## 6. Visualisation comparative des 3 itinéraires

In [ ]:
ROUTE_STYLES = {
    'shortest': {'color': 'blue',   'weight': 5, 'label': '🔵 Plus court'},
    'safest':   {'color': 'green',  'weight': 5, 'label': '🟢 Plus sûr'},
    'balanced': {'color': 'orange', 'weight': 5, 'label': '🟠 Équilibré'},
}

m2 = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles='CartoDB positron')

# Fond : heatmap de risque (grisé)
for u, v, k, data in G.edges(keys=True, data=True):
    risk = kde_result.risk_map.get((u, v, k), 0.0)
    opacity = 0.1 + 0.3 * risk
    r = int(200 * risk)
    folium.PolyLine(
        locations=[[G.nodes[u]['y'], G.nodes[u]['x']], [G.nodes[v]['y'], G.nodes[v]['x']]],
        color=f'#{r:02x}0000', weight=1, opacity=opacity
    ).add_to(m2)

# Les 3 itinéraires
for route in [pareto.shortest, pareto.safest, pareto.balanced]:
    if route is None:
        continue
    style = ROUTE_STYLES[route.route_type]
    coords = [[G.nodes[nid]['y'], G.nodes[nid]['x']] for nid in route.path if nid in G.nodes]
    if len(coords) >= 2:
        folium.PolyLine(
            locations=coords,
            color=style['color'], weight=style['weight'], opacity=0.9,
            tooltip=(
                f"{style['label']}\n"
                f"Distance : {route.total_distance_m:.0f}m\n"
                f"Risque : {route.total_risk:.3f}\n"
                f"Temps : {route.estimated_time_min:.1f} min\n"
                f"Confort : {route.comfort_score:.2f}"
            )
        ).add_to(m2)

# Marqueurs source / destination
src_coords = [G.nodes[SOURCE]['y'], G.nodes[SOURCE]['x']]
tgt_coords = [G.nodes[TARGET]['y'], G.nodes[TARGET]['x']]
folium.Marker(src_coords, tooltip='Départ', icon=folium.Icon(color='red', icon='play')).add_to(m2)
folium.Marker(tgt_coords, tooltip='Arrivée', icon=folium.Icon(color='red', icon='stop')).add_to(m2)

display(m2)

## 7. Comparaison graphique des 3 itinéraires

In [ ]:
routes = [r for r in [pareto.shortest, pareto.safest, pareto.balanced] if r is not None]
labels = [r.route_type for r in routes]
colors = ['#2196F3', '#4CAF50', '#FF9800']

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle(f'SafeRoute — Comparaison des 3 itinéraires ({CITY.replace("_", " ").title()})',
             fontsize=14, fontweight='bold')

metrics = [
    ('Distance (m)',    [r.total_distance_m for r in routes]),
    ('Risque cumulé',   [r.total_risk for r in routes]),
    ('Temps estimé (min)', [r.estimated_time_min for r in routes]),
]

for ax, (title, values) in zip(axes, metrics):
    bars = ax.bar(labels, values, color=colors, edgecolor='white', linewidth=1.5)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel(title)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
                f'{val:.1f}', ha='center', va='bottom', fontsize=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../data/saferoute_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Graphique sauvegardé dans data/saferoute_comparison.png')

## 8. Frontière de Pareto

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for route, color in zip(routes, colors):
    ax.scatter(route.total_distance_m, route.total_risk,
               s=200, c=color, zorder=5, label=route.route_type)
    ax.annotate(f'  {route.route_type}\n  {route.total_distance_m:.0f}m | risk={route.total_risk:.2f}',
                (route.total_distance_m, route.total_risk),
                fontsize=9, va='center')

# Ligne de Pareto
pts = sorted([(r.total_distance_m, r.total_risk) for r in routes])
ax.plot([p[0] for p in pts], [p[1] for p in pts],
        'k--', alpha=0.4, linewidth=1, label='Frontière de Pareto')

ax.set_xlabel('Distance totale (m)', fontsize=12)
ax.set_ylabel('Risque cumulé', fontsize=12)
ax.set_title('Frontière de Pareto — Distance vs Sécurité', fontsize=13, fontweight='bold')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../data/pareto_frontier.png', dpi=150, bbox_inches='tight')
plt.show()
print('Frontière de Pareto sauvegardée dans data/pareto_frontier.png')

## 9. Résumé

| Itinéraire | Distance | Risque | Temps | Confort |
|------------|----------|--------|-------|---------|
| 🔵 Plus court | min | élevé | min | faible |
| 🟢 Plus sûr | élevé | min | élevé | max |
| 🟠 Équilibré | moyen | moyen | moyen | moyen |

**Formule** : `C = 1.0·Distance + 500·Risque - 200·Familiarité`

**Algorithme** : A*pex ε-approximé (ε=0.1) — garantit des solutions à 10% de l'optimal Pareto exact.